In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 28 samples and 15003 columns.
First few rows of the data:


,RFS (months),RFS Censor,Patient ID,RNA-Seq ID,Date of Esophageal Surgery,Relapse,Date of Relapse or Censoring,Date of Death,B cells naive,B cells memory,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,26.1184,0,B012-0001,SL462108,2016-04-01,No,2018-06-04,NaN,0.022107,0.00000,...,3.662069,2.487979,3.810792,3.733372,1.791442,2.494363,4.520041,4.071395,2.996526,3.639040
1,24.5724,0,B012-0004,SL462135,2016-06-10,No,2018-06-27,NaN,0.019443,0.00000,...,2.473485,1.498157,2.663531,2.518551,1.111765,1.356405,3.285345,3.232526,2.252723,2.644813
2,14.0461,1,B012-0007,SL446519,2016-08-01,Yes,2017-10-02,NaN,0.037155,0.00000,...,4.116829,3.025456,4.343368,4.213837,2.025170,2.679345,4.979327,4.475582,3.544341,4.142523
3,17.1382,0,B012-0008,SL462109,2016-10-14,No,2018-03-19,NaN,0.010599,0.00000,...,2.842831,1.888814,3.229611,3.045209,1.598104,2.053633,3.996419,3.611956,2.599311,3.154038
4,24.9671,1,B012-0011,SL446521,2016-12-19,No,2019-01-17,2019-01-17,0.030652,0.00247,...,2.953496,2.304290,3.511578,3.084574,1.602120,2.004306,4.262906,4.016896,3.214875,3.300263


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for RFS (months) or RFS Censor: 0
Number of samples with complete data for both variables: 28
Number of censored samples (event_var == 0.0): 15
Number of events (event_var == 1.0): 13
Censored ratio (among complete): 15 / 28 = 0.536 (53.6%)
Skipping stats for OS: columns not in preprocessed table ('unknown', 'unknown').
Skipping stats for OS after relapse: columns not in preprocessed table ('unknown', 'unknown').
Updated stats for 1 of 3 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE183924/description.json
